In [1]:
from medmnist import BloodMNIST
import torch
from torch import nn, utils, Generator
import torch.nn.functional as F
import torchvision
from tqdm.notebook import trange, tqdm
import plotly.express as px
from typing import Sequence
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"
import plotly.colors as pc

In [2]:
import plotly.io as pio
pio.get_chrome()

PosixPath('/home/roman/repos/learning-ml/.venv/lib/python3.12/site-packages/choreographer/cli/browser_exe/chrome-linux64/chrome')

In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0)) 
EPOCHS = 30

True
NVIDIA GeForce RTX 3080 Ti


## Dataset Preparation

[MedMNIST](https://doi.org/10.1038/s41597-022-01721-8) is a collection of 18x standardized datasets for 2D and 3D biomedical image classification. 

It contains multiple size options: 28 (MNIST-Like), 64, 128, and 224.

In [4]:
prepare_dataset = lambda x: BloodMNIST(x, torchvision.transforms.ToTensor(), lambda x:x.squeeze(), download=True, root="/var/tmp", size=128)
ds_train, ds_val, ds_test = [prepare_dataset(x) for x in ['train', 'val', 'test']]

random_generator = Generator().manual_seed(42)
prepare_dataloader = lambda x: utils.data.DataLoader(x, 64, True, generator=random_generator)

dl_train, dl_val, dl_test = [prepare_dataloader(x) for x in [ds_train, ds_val, ds_test]]

classes, label_lookup = len(ds_train.info['label']), ds_train.info['label']
label_lookup[str(3)] = "immature granulocytes" # I wanted a shorter name

ds_train

Dataset BloodMNIST of size 128 (bloodmnist_128)
    Number of datapoints: 11959
    Root location: /var/tmp
    Split: train
    Task: multi-class
    Number of channels: 3
    Meaning of labels: {'0': 'basophil', '1': 'eosinophil', '2': 'erythroblast', '3': 'immature granulocytes', '4': 'lymphocyte', '5': 'monocyte', '6': 'neutrophil', '7': 'platelet'}
    Number of samples: {'train': 11959, 'val': 1712, 'test': 3421}
    Description: The BloodMNIST is based on a dataset of individual normal cells, captured from individuals without infection, hematologic or oncologic disease and free of any pharmacologic treatment at the moment of blood collection. It contains a total of 17,092 images and is organized into 8 classes. We split the source dataset with a ratio of 7:1:2 into training, validation and test set. The source images with resolution 3×360×363 pixels are center-cropped into 3×200×200, and then resized into 3×28×28.
    License: CC BY 4.0

In [5]:
ex = next(iter(dl_train))
print(ex[0].shape, ex[1].shape)

img = torch.einsum("nchw->nhwc",ex[0][:10])
fig = px.imshow(img, binary_string=True, facet_col=0, facet_col_wrap=5, facet_col_spacing=0.01, facet_row_spacing=0.08)

item_map={f'{i}':label_lookup[str(key.item())] for i, key in enumerate(ex[1][:10])}
fig.for_each_annotation(lambda a: a.update(text=item_map[a.text.split("=")[1]])) 

fig.update_layout(
        # title={"text": "N", "x": 0.5},
        height=600,
        width=1500,
        margin=dict(l=20, r=20, t=20, b=20),
    )

fig.write_image(f"figures/sample_images_blood_mnist.svg")

fig.show()

torch.Size([64, 3, 128, 128]) torch.Size([64])


## Model Definitions

In [6]:
class NamedModule(nn.Module):
    def set_name(self, name:str):
        self.override_name = name
        return self

    def name(self) -> str:
        if hasattr(self, "override_name"):
            return self.override_name
        name_attr = getattr(self, "_name", None)
        if callable(name_attr):
            return name_attr() # type: ignore
        elif isinstance(name_attr, str):
            return name_attr
        else:
            raise NotImplementedError
    
class SoftmaxRegression(NamedModule):
    def __init__(self, outfeatures: int, bias = True):
        super().__init__()
        self.f = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(outfeatures, bias)
        )
    
    def name(self) -> str:
        return "softmax"

    def forward(self, x):
        return self.f(x)

class CNNBlock(NamedModule):
    def __init__(self, out_channels:int, conv_layers=4, skip_frequency=2, batch_norm=True):
        super().__init__()
        self.conv_layers = conv_layers
        self.features = nn.ModuleList()
        self.residuals = nn.ModuleList()
        self.skip_frequency = skip_frequency
        
        for i in range(conv_layers):
            self.features.append(nn.Sequential(nn.LazyConv2d(out_channels, 3, padding=1), *([nn.GroupNorm(1, out_channels)] if batch_norm else []), nn.ReLU()))
            if self._should_end_skip(i):
                self.residuals.append(nn.LazyConv2d(out_channels, 1))

    def _has_skips(self):
        return self.skip_frequency != 0

    def _should_start_skip(self, i):
        return self._has_skips() and (i%self.skip_frequency == 0 and (i+self.skip_frequency) <= self.conv_layers)
    
    def _should_end_skip(self, i):
        return self._has_skips() and ((i+1)%self.skip_frequency == 0)

    def forward(self, x):
        identity = 0
        for i, conv in enumerate(self.features):
            if self._should_start_skip(i):
                identity = self.residuals[i//self.skip_frequency](x)
            x = conv(x)
            if self._should_end_skip(i):
                x = x + identity
        return x

class CNN(NamedModule):
    r"""Construct VGG/ResNet hybrid CNN

    Core architecture of VGG remains the same, with the addition of skip
    connections within each block.

    Standard features of ResNet such as strides conv instead of max pool,
    global average pooling instead of FC layers and bottleneck blocks have
    not been included.
    
    Args:
        arch: List of tuples. Each tuple contains number of convolutions in layer, out channels of layer,
            residual frequency in layer. Set residual frequency to 0 to disable residuals.
        in_channels: Number of channels in the input image.
        classes: Number of logits.
        dropout_rate: Dropout used in the classifier layer.
    """

    def __init__(self, arch: list[tuple[int,int,int]], classes=10, dropout_rate=0.0, batch_norm=True):
        super().__init__()

        self.max_pool = nn.MaxPool2d(2)
        self.dropout_rate = dropout_rate
        self.features = nn.ModuleList()
        self.arch = arch

        layers = []
        for (num_convs, out_ch, residuals) in self.arch:
            layers.append(CNNBlock(out_ch, num_convs, residuals, batch_norm))
            layers.append(self.max_pool)

        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.LazyLinear(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.LazyLinear(classes),
        )

        # ResNet style classifier.
        # self.classifier = nn.Sequential(
        #     nn.AdaptiveAvgPool2d(1),
        #     nn.Flatten(),
        #     nn.LazyLinear(classes),
        # )

    def _name(self):
        convolutions = sum([k[0] for k in self.arch])
        skip_frequency = self.arch[0][2]
        n = f"cnn"
        n += f"-{convolutions}"
        if self.dropout_rate > 0:
            n += f"-d-{self.dropout_rate}"
        if skip_frequency > 0:
            n += f"-s-{skip_frequency}"
        return n

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [7]:
class Record():
    def __init__(self):
        self.store = {}
        self.epoch_store = {}

    def add(self, name, loss, logits, targets):
        self.store.setdefault(f"{name}-loss", []).append(loss)
        acc = (logits.argmax(1) == targets).float().mean().item()
        self.store.setdefault(f"{name}-acc", []).append(acc)

    def compute(self):
        for key, values in self.store.items():
            self.epoch_store.setdefault(key, []).append(
                sum(values) / len(values)
            )
        self.store = {}

    def clear(self):
        self.store = {}
        self.epoch_store = {}

In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re


def plot_comparison(metrics: Record, title="Model Comparison", splits=("val", "train"), measures=("acc", "loss"), split_subplots=False, file_name=""):
    titles = {
        "acc": "Accuracy",
        "loss": "Loss",
        "val": "Validation",
        "train": "Train"
    }

    if split_subplots:
        rows = len(splits)
        cols = len(measures)
        subplot_titles = [f"{titles[s]} {titles[m]} vs Epoch" for s in splits for m in measures]
    else:
        rows = 1
        cols = len(measures)
        subplot_titles = [f"{titles[m]} vs Epoch" for m in measures]

    fig = make_subplots(rows=rows, cols=cols, subplot_titles=subplot_titles, horizontal_spacing=0.04, vertical_spacing=0.08)
    models = set(k.rsplit("-", 2)[0] for k in metrics.epoch_store.keys())
    colors = pc.qualitative.Plotly

    def natural_sort_key(s):
        return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]

    trace_idx = 0
    shown = set()
    for name in sorted(models, key=natural_sort_key):
        if split_subplots:
            color = colors[trace_idx % len(colors)]
            trace_idx += 1
            for split in splits:
                for m_idx, measure in enumerate(measures):
                    key = f"{name}-{split}-{measure}"
                    values = metrics.epoch_store.get(key)
                    if values is None:
                        continue
                    epochs = list(range(len(values)))
                    show = name not in shown
                    shown.add(name)
                    fig.add_trace(
                        go.Scatter(x=epochs, y=values, name=name, legendgroup=name, showlegend=show, line=dict(color=color)),
                        row=splits.index(split) + 1, col=m_idx + 1
                    )
        else:
            for split in splits:
                color = colors[trace_idx % len(colors)]
                trace_idx += 1
                label = f"{name} ({split})" if len(splits) > 1 else name
                for m_idx, measure in enumerate(measures):
                    key = f"{name}-{split}-{measure}"
                    values = metrics.epoch_store.get(key)
                    if values is None:
                        continue
                    epochs = list(range(len(values)))
                    show = label not in shown
                    shown.add(label)
                    fig.add_trace(
                        go.Scatter(x=epochs, y=values, name=label, legendgroup=label, showlegend=show, line=dict(color=color)),
                        row=1, col=m_idx + 1
                    )

    fig.update_xaxes(title_text="Epoch")
    for m_idx, measure in enumerate(measures):
        for r in range(1, rows + 1):
            fig.update_yaxes(title_text=titles[measure], row=r, col=m_idx + 1)

    fig.update_layout(
        title={"text": title, "x": 0.5},
        showlegend=len(models) > 1 or len(splits) > 1,
        height=600 * rows,
        width=1500
    )
    if file_name:
        fig.write_image(f"figures/{file_name}.svg")
    fig.show()
# plot_comparison(softmax_metrics, "Softmax", file_name="Softmax")
# plot_comparison(softmax_metrics, "Softmax", splits=("val","train"), measures=("acc","loss"))
# plot_comparison(metrics_batchnorm_off, "CNN Comparison Normalisation Off (Validation)", splits=("val",))
# plot_comparison(metrics_batchnorm_off, "CNN Comparison Normalisation Off (Train)", splits=("train",))
# plot_comparison(metrics_batchnorm_off, "CNN Comparison Normalisation Off", splits=("train","val"), split_subplots=True)

In [9]:
def eval_model(model: NamedModule, metric_store: Record, device: torch.device, epochs=EPOCHS, lr=1e-3, batch_size=64):
    model.to(device)
    dl = utils.data.DataLoader(ds_train, batch_size, True, generator=Generator().manual_seed(42))
    trainer = torch.optim.Adam(model.parameters(), lr=lr)
    loss = nn.CrossEntropyLoss()
    with trange(epochs) as t:
        t.set_description(model.name())
        for i in t:
            model.train()
            for X, y in tqdm(dl, leave=False, desc="train"):
                X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
                out = model(X)
                l = loss(out, y)
                trainer.zero_grad()
                l.backward()
                trainer.step()
                metric_store.add(f"{model.name()}-train", l.item(), out.detach(), y)

            model.eval()
            with torch.no_grad():
                for X, y in tqdm(dl_val, leave=False, desc="val"):
                    X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
                    
                    out = model(X)
                    l = loss(out, y)
                    metric_store.add(f"{model.name()}-val", l.item(), out.detach(), y)

            metric_store.compute()
            t.set_description(f"{model.name()} ({max(metric_store.epoch_store[f"{model.name()}-val-acc"])*100:.0f}%)")

    model.cpu()
    torch.cuda.empty_cache()

def eval_models(models: Sequence[NamedModule], device: torch.device, epochs=EPOCHS, lr=1e-3, batch_size=64):
    metric_store = Record()
    for model in models:
        eval_model(model, metric_store, device, epochs, lr=lr, batch_size=batch_size)
    return metric_store

def test_model(model: nn.Module, device: torch.device):
    metrics = Record()
    model.to(device)
    model.eval()
    loss = nn.CrossEntropyLoss()
    with torch.no_grad():
        for X, y in tqdm(dl_test, leave=False, desc="test"):
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
            
            out = model(X)
            l = loss(out, y)
            metrics.add(f"", l.item(), out.detach(), y)
    metrics.compute()
    return (metrics.epoch_store["-loss"][0], metrics.epoch_store["-acc"][0])

def test_models(models: Sequence[NamedModule], device: torch.device):
    out = "Test summary metrics:\n"
    for model in models:
        test_loss, test_acc = test_model(model, device)
        out += f"  {model.name()}: loss={test_loss:.2f}, acc={test_acc*100:.2f}%\n"
    print(out)

In [10]:
device = torch.device('cuda')

## Softmax

In [10]:
linear = SoftmaxRegression(classes)
softmax_metrics = eval_models([linear], device)

  0%|          | 0/50 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

In [52]:
test_models([linear], device)
plot_comparison(softmax_metrics, "Softmax performance on BloodMNIST (128x128)", file_name="softmax_bloodmnist")

test:   0%|          | 0/54 [00:00<?, ?it/s]

Test summary metrics:
  softmax: loss=2.15, acc=69.47%



## Convolutional Neural Network

### Batchnorm off

In [12]:
model2_norm_off = CNN(arch=[(1,32,0), (1,64,0)], classes=classes, batch_norm=False) # 1+1
model8_norm_off = CNN(arch=[(3,32,0), (3,64,0), (2,128,0)], classes=classes, batch_norm=False) # 3+3+2
model16_norm_off = CNN(arch=[(6,32,0), (5,64,0), (5,128,0)], classes=classes, batch_norm=False) # 6+5+5
model32_norm_off = CNN(arch=[(8,32,0), (8,64,0), (8,128,0), (8,256,0)], classes=classes, batch_norm=False) # 8+8+8+8

models_batchnorm_off = [model2_norm_off, model8_norm_off, model16_norm_off, model32_norm_off]
metrics_batchnorm_off = eval_models(models_batchnorm_off, device, EPOCHS)

  0%|          | 0/50 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

In [31]:
test_models(models_batchnorm_off, device)
plot_comparison(metrics_batchnorm_off, "CNN Comparison Normalisation Off", splits=("val","train"), split_subplots=True)

test:   0%|          | 0/54 [00:00<?, ?it/s]

test:   0%|          | 0/54 [00:00<?, ?it/s]

test:   0%|          | 0/54 [00:00<?, ?it/s]

test:   0%|          | 0/54 [00:00<?, ?it/s]

Test summary metrics:
  cnn-2: loss=0.41, acc=93.98%
  cnn-8: loss=2.00, acc=19.48%
  cnn-16: loss=2.00, acc=19.34%
  cnn-32: loss=2.00, acc=19.48%



### Batchnorm on

In [ ]:
model2 = CNN(arch=[(1,32,0), (1,64,0)], classes=classes, batch_norm=True) # 1+1
model8 = CNN(arch=[(3,32,0), (3,64,0), (2,128,0)], classes=classes, batch_norm=True) # 3+3+2
model16 = CNN(arch=[(6,32,0), (5,64,0), (5,128,0)], classes=classes, batch_norm=True) # 6+5+5
model32 = CNN(arch=[(8,32,0), (8,64,0), (8,128,0), (8,256,0)], classes=classes, batch_norm=True) # 8+8+8+8

models_batchnorm_on = [model2, model8, model16, model32]
metrics_batchnorm_on = eval_models(models_batchnorm_on, device, EPOCHS)

  0%|          | 0/50 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

In [73]:
import copy
filtered_metrics = copy.deepcopy(metrics_batchnorm_on)

In [74]:
for key in filtered_metrics.epoch_store:
    filtered_metrics.epoch_store[key] = filtered_metrics.epoch_store[key][0:30]

In [76]:
print(filtered_metrics.epoch_store)

{'cnn-2-train-loss': [1.3970724007981346, 0.3854342294249305, 0.29036313143962206, 0.20591237732672438, 0.14952328424641792, 0.13781427262340956, 0.11423655569722707, 0.11012116887850716, 0.09004680219450019, 0.0654742723980789, 0.077681782969179, 0.0459443871828324, 0.04886216117457591, 0.04060584179407115, 0.051269392744715224, 0.03243883991571029, 0.018084611506188876, 0.026386128123295566, 0.026858944931800726, 0.05744435665207631, 0.021967671135500234, 0.02052824726852512, 0.01127318366323934, 0.027914633135985828, 0.019612192833764395, 0.0045810843066014795, 0.00015146615186602182, 5.508039209737891e-05, 3.557374177027649e-05, 2.7365250727287053e-05], 'cnn-2-train-acc': [0.6033057850949904, 0.8581657145112593, 0.8953193362383919, 0.9257064292775118, 0.9477774060983709, 0.9513429750733197, 0.9598383565637517, 0.9591562346341138, 0.9687925376356604, 0.9768002549594736, 0.9725935825689591, 0.9837627611695764, 0.9845983226669026, 0.9858516649128919, 0.9819245258754588, 0.989291139783

In [ ]:
test_models(models_batchnorm_on, device)
plot_comparison(filtered_metrics, "CNN Performance Comparison on BloodMNIST (128x128)", splits=("val","train"), split_subplots=True, file_name="cnn_comparison_bloodmnist")

test:   0%|          | 0/54 [00:00<?, ?it/s]

test:   0%|          | 0/54 [00:00<?, ?it/s]

test:   0%|          | 0/54 [00:00<?, ?it/s]

test:   0%|          | 0/54 [00:00<?, ?it/s]

Test summary metrics:
  cnn-2: loss=0.33, acc=95.94%
  cnn-8: loss=0.53, acc=91.67%
  cnn-16: loss=0.67, acc=91.15%
  cnn-32: loss=0.27, acc=91.62%



### Improving Deeper Model

In [65]:
model16 = CNN(arch=[(6,32,0), (5,64,0), (5,128,0)], classes=classes, batch_norm=True) # 6+5+5
model32 = CNN(arch=[(8,32,0), (8,64,0), (8,128,0), (8,256,0)], classes=classes, batch_norm=True) # 8+8+8+8
model16improved = CNN(arch=[(6,32,2), (5,64,2), (5,128,2)], classes=classes, dropout_rate=0.5, batch_norm=True)
model32improved = CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, batch_norm=True)

models_improvement_comparison = [model16, model32, model16improved, model32improved]
metrics_improvement_comparison = eval_models(models_improvement_comparison, device, EPOCHS)

  0%|          | 0/30 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

In [67]:
print(metrics_improvement_comparison.epoch_store)

{'cnn-16-train-loss': [1.2372568421822818, 0.6701305454427545, 0.5109129270767783, 0.4414873062608076, 0.36220868449797605, 0.29917201128872956, 0.2593483887812033, 0.2304429373240726, 0.18367010990486426, 0.16372867469441763, 0.14001291217332218, 0.10717028739020468, 0.09122847191451545, 0.07853374884707882, 0.09592421256742853, 0.06179150920311477, 0.04415501984655558, 0.043562268877506534, 0.034707971568403634, 0.06464298555121961, 0.03765692448468859, 0.04068443304946258, 0.03008767800096841, 0.04381755282504753, 0.016509647033458082, 0.008506850518958884, 0.047558970167963045, 0.03461642514006608, 0.02435415476046373, 0.017533323634322483], 'cnn-16-train-acc': [0.5398881865695199, 0.7497235048900951, 0.8111813320195611, 0.8345314777471164, 0.8632899854272444, 0.8881335073613865, 0.9033695915166069, 0.9150659333575856, 0.9330031597678037, 0.9383218884468079, 0.9513991854407571, 0.9629299341038586, 0.9678597469380833, 0.9712430114414603, 0.9673021996722502, 0.9784850510046443, 0.985

In [68]:
test_models(models_improvement_comparison, device)
plot_comparison(metrics_improvement_comparison, "CNN-I Performance Comparison on BloodMNIST (128x128)", splits=("val","train"), split_subplots=True, file_name="cnn_comparison_improved_bloodmnist")

test:   0%|          | 0/54 [00:00<?, ?it/s]

test:   0%|          | 0/54 [00:00<?, ?it/s]

test:   0%|          | 0/54 [00:00<?, ?it/s]

test:   0%|          | 0/54 [00:00<?, ?it/s]

Test summary metrics:
  cnn-16: loss=0.64, acc=88.92%
  cnn-32: loss=0.55, acc=86.74%
  cnn-16-d-0.5-s-2: loss=0.17, acc=95.13%
  cnn-32-d-0.5-s-2: loss=0.19, acc=95.21%



### Learning Rate Comparison

In [11]:
metric_learning_rates = Record()

In [12]:
model_0 = CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, batch_norm=True).set_name(f"lr-{0.000001}")
eval_model(model_0, metric_learning_rates, device, EPOCHS, lr=0.000001)

  0%|          | 0/30 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

In [13]:
model_1 = CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, batch_norm=True).set_name(f"lr-{0.0001}")
eval_model(model_1, metric_learning_rates, device, EPOCHS, lr=1e-4)

  0%|          | 0/30 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

In [14]:
model_2 = CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, batch_norm=True).set_name(f"lr-{0.001}")
eval_model(model_2, metric_learning_rates, device, EPOCHS, lr=1e-3)

  0%|          | 0/30 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

In [15]:
model_3 = CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, batch_norm=True).set_name(f"lr-{0.01}")
eval_model(model_3, metric_learning_rates, device, EPOCHS, lr=1e-2)

  0%|          | 0/30 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

In [16]:
model_4 = CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, batch_norm=True).set_name(f"lr-{1.0}")
eval_model(model_4, metric_learning_rates, device, EPOCHS, lr=1.0)

  0%|          | 0/30 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

In [17]:
test_models([model_0, model_1, model_2, model_3, model_4], device)

test:   0%|          | 0/54 [00:00<?, ?it/s]

test:   0%|          | 0/54 [00:00<?, ?it/s]

test:   0%|          | 0/54 [00:00<?, ?it/s]

test:   0%|          | 0/54 [00:00<?, ?it/s]

test:   0%|          | 0/54 [00:00<?, ?it/s]

Test summary metrics:
  lr-1e-06: loss=0.19, acc=93.48%
  lr-0.0001: loss=0.15, acc=95.69%
  lr-0.001: loss=2.00, acc=19.52%
  lr-0.01: loss=2.00, acc=19.45%
  lr-1.0: loss=116353237962637841477461344256.00, acc=16.58%



In [18]:
plot_comparison(metric_learning_rates, "CNN Learning Rate Comparison", splits=("val","train"), split_subplots=True, file_name="cnn_learning_rate_2_blood_mnist")

## Minibatch Size Comparison

In [19]:
metric_minibatch_sizes = Record()

In [20]:
b_model_1 = CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, batch_norm=True).set_name(f"batch-1")
eval_model(b_model_1, metric_minibatch_sizes, device, EPOCHS, lr=1e-3, batch_size=1)

  0%|          | 0/30 [00:00<?, ?it/s]

train:   0%|          | 0/11959 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/11959 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/11959 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/11959 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

In [21]:
b_model_2 = CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, batch_norm=True).set_name(f"batch-2")
eval_model(b_model_2, metric_minibatch_sizes, device, EPOCHS, lr=1e-3, batch_size=8)

  0%|          | 0/30 [00:00<?, ?it/s]

train:   0%|          | 0/1495 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/1495 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/1495 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/1495 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

In [22]:
b_model_3 = CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, batch_norm=True).set_name(f"batch-3")
eval_model(b_model_3, metric_minibatch_sizes, device, EPOCHS, lr=1e-3, batch_size=16)

  0%|          | 0/30 [00:00<?, ?it/s]

train:   0%|          | 0/748 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/748 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/748 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/748 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

In [23]:
b_model_4 = CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, batch_norm=True).set_name(f"batch-4")
eval_model(b_model_4, metric_minibatch_sizes, device, EPOCHS, lr=1e-3, batch_size=64)

  0%|          | 0/30 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/187 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

In [24]:
b_model_5 = CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, batch_norm=True).set_name(f"batch-5")
eval_model(b_model_5, metric_minibatch_sizes, device, EPOCHS, lr=1e-3, batch_size=256)

  0%|          | 0/30 [00:00<?, ?it/s]

train:   0%|          | 0/47 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/47 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

...

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/47 [00:00<?, ?it/s]

val:   0%|          | 0/27 [00:00<?, ?it/s]

train:   0%|          | 0/47 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [25]:
print(metric_minibatch_sizes.epoch_store)

{'batch-1-train-loss': [43649.152068372496, 9223.375701144414, 78272.86289691964, 11774.582673549503, 2.003723416323247, 104612.31429558258, 364157.3416268055, 2.00361583239658, 13357.973547697884, 41196.556897309034, 56199.95843331442, 2.0038058063598947, 2.00367183380875, 2.0036752882166464, 2.003676559825597, 2.003597465942253, 27053.169547437326, 1924.986671073798, 2.003849716417386, 2238.471801164268, 52398.468218468304, 2.0034602370406884, 2.003698537397349, 5309.696756462267, 2.0036345342796715, 2.003756401002362, 2.0036309008933655, 49738.90722471654, 2.0038164902494966, 2.0037762785906854], 'batch-1-train-acc': [0.18111882264403378, 0.19282548708085961, 0.1948323438414583, 0.19374529642946733, 0.19458148674638348, 0.19315996320762605, 0.191654820637177, 0.19382891546115896, 0.19416339158792542, 0.19357805836608413, 0.19449786771469185, 0.19315996320762605, 0.1948323438414583, 0.19374529642946733, 0.1939961535245422, 0.1902332970984196, 0.1921565348273267, 0.1934944393343925, 0

In [26]:
plot_comparison(metric_minibatch_sizes, "CNN Minibatch Size Comparison", splits=("val","train"), split_subplots=True, file_name="cnn_minibatch_blood_mnist")

In [20]:
# X, y = next(iter(dl_test))
# m = nn.Softmax(dim=1)
# # If model on gpu
# # X_gpu = X.to(device, non_blocking=True)
# # y_hat_probs = m(model(X_gpu)).cpu()

# y_hat_probs = m(model32improved(X))

# y_hat = y_hat_probs.argmax(axis=1)

# img = torch.einsum("nchw->nhwc",X)
# fig = px.imshow(img, binary_string=True, facet_col=0, facet_col_wrap=8, height=2000, facet_col_spacing=0.01, facet_row_spacing=0.01)

# for i, label in enumerate(y_hat):
#     text = f"{label_lookup[str(label.item())]} ({y_hat_probs[i][label]*100:.2f}%)"
#     if label != y[i]:
#         text = f"<span style='color:red'>{text}</span>"
    
#     item_map[f'{i}'] = text

# fig.for_each_annotation(lambda a: a.update(text=item_map[a.text.split("=")[1]])) 

# fig.show()